# Notebook 15: Proyecto Integrador - Sistema Completo de Calidad de Datos

## Objetivos del Proyecto

En este proyecto integrador aplicaras todos los conceptos aprendidos en el taller para construir un sistema completo de calidad de datos que incluye:

1. Validacion de multiples fuentes de datos (CSV, Parquet, Base de Datos)
2. Implementacion de las 5 dimensiones de calidad
3. Checkpoints automatizados
4. Data Docs personalizados
5. Integracion con CI/CD
6. Alertas y notificaciones

## Escenario del Proyecto

Trabajaras con un pipeline de datos de e-commerce que procesa:
- Datos de clientes (CSV)
- Transacciones de ventas (Parquet)
- Inventario de productos (PostgreSQL)

Tu mision es garantizar la calidad de estos datos antes de que lleguen al data warehouse.

## Parte 1: Configuracion del Proyecto

### 1.1 Importaciones y Configuracion Inicial

In [ ]:
import great_expectations as gx
from great_expectations.checkpoint import Checkpoint
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from pathlib import Path

print('Great Expectations version:', gx.__version__)

### 1.2 Crear Datos de Ejemplo

Primero crearemos datasets de ejemplo que simulan datos reales de e-commerce.

In [ ]:
# Crear directorio para datos
data_dir = Path('../data/proyecto_integrador')
data_dir.mkdir(parents=True, exist_ok=True)

# Dataset 1: Clientes (CSV)
np.random.seed(42)
n_customers = 1000

customers_df = pd.DataFrame({
    'customer_id': range(1, n_customers + 1),
    'email': [f'customer{i}@example.com' for i in range(1, n_customers + 1)],
    'nombre': [f'Cliente {i}' for i in range(1, n_customers + 1)],
    'pais': np.random.choice(['Mexico', 'Colombia', 'Argentina', 'Chile', 'Peru'], n_customers),
    'fecha_registro': pd.date_range(start='2023-01-01', periods=n_customers, freq='8H'),
    'edad': np.random.randint(18, 70, n_customers),
    'segmento': np.random.choice(['Premium', 'Standard', 'Basic'], n_customers, p=[0.2, 0.5, 0.3])
})

# Introducir algunos problemas de calidad intencionalmente
customers_df.loc[10:15, 'email'] = None  # Valores nulos
customers_df.loc[20, 'customer_id'] = customers_df.loc[21, 'customer_id']  # Duplicado
customers_df.loc[30:35, 'edad'] = -1  # Valores invalidos

customers_df.to_csv(data_dir / 'customers.csv', index=False)
print(f'Dataset de clientes creado: {len(customers_df)} registros')

In [ ]:
# Dataset 2: Transacciones (Parquet)
n_transactions = 5000

transactions_df = pd.DataFrame({
    'transaction_id': range(1, n_transactions + 1),
    'customer_id': np.random.randint(1, n_customers + 1, n_transactions),
    'product_id': np.random.randint(1, 101, n_transactions),
    'fecha_transaccion': pd.date_range(start='2024-01-01', periods=n_transactions, freq='10min'),
    'cantidad': np.random.randint(1, 10, n_transactions),
    'precio_unitario': np.round(np.random.uniform(10, 500, n_transactions), 2),
    'descuento': np.round(np.random.uniform(0, 0.3, n_transactions), 2),
    'metodo_pago': np.random.choice(['Tarjeta', 'Efectivo', 'Transferencia'], n_transactions)
})

# Calcular monto total
transactions_df['monto_total'] = (
    transactions_df['cantidad'] * 
    transactions_df['precio_unitario'] * 
    (1 - transactions_df['descuento'])
).round(2)

# Introducir problemas de calidad
transactions_df.loc[100:110, 'cantidad'] = 0  # Cantidad cero
transactions_df.loc[200:205, 'precio_unitario'] = -50  # Precios negativos
transactions_df.loc[300, 'monto_total'] = 999999  # Monto inconsistente

transactions_df.to_parquet(data_dir / 'transactions.parquet', index=False)
print(f'Dataset de transacciones creado: {len(transactions_df)} registros')

### 1.3 Inicializar File Context

In [ ]:
# Crear directorio para GX
gx_dir = Path('../gx_proyecto_integrador')

# Inicializar File Context
context = gx.get_context(mode='file', project_root_dir=str(gx_dir))
print('File Context inicializado correctamente')

## Parte 2: Configuracion de Data Sources

### 2.1 Data Source para Clientes (CSV)

In [ ]:
# Configurar data source para CSV
datasource_customers = context.sources.add_pandas(
    name='customers_datasource'
)

# Agregar asset
asset_customers = datasource_customers.add_csv_asset(
    name='customers_csv',
    filepath_or_buffer=str(data_dir / 'customers.csv')
)

# Crear batch request
batch_request_customers = asset_customers.build_batch_request()
print('Data source de clientes configurado')

### 2.2 Data Source para Transacciones (Parquet)

In [ ]:
# Configurar data source para Parquet
datasource_transactions = context.sources.add_pandas(
    name='transactions_datasource'
)

# Agregar asset
asset_transactions = datasource_transactions.add_parquet_asset(
    name='transactions_parquet',
    filepath_or_buffer=str(data_dir / 'transactions.parquet')
)

# Crear batch request
batch_request_transactions = asset_transactions.build_batch_request()
print('Data source de transacciones configurado')

## Parte 3: Expectation Suites por Dimension de Calidad

### 3.1 Suite de Completitud - Clientes

In [ ]:
# Crear suite de completitud
suite_completitud = context.add_expectation_suite(
    expectation_suite_name='customers_completitud_suite'
)

# Crear validator
validator_customers = context.get_validator(
    batch_request=batch_request_customers,
    expectation_suite_name='customers_completitud_suite'
)

# Expectativas de completitud
validator_customers.expect_column_values_to_not_be_null(
    column='customer_id',
    meta={'dimension': 'Completitud', 'criticidad': 'Alta'}
)

validator_customers.expect_column_values_to_not_be_null(
    column='email',
    mostly=0.95,
    meta={'dimension': 'Completitud', 'criticidad': 'Alta'}
)

validator_customers.expect_column_values_to_not_be_null(
    column='nombre',
    meta={'dimension': 'Completitud', 'criticidad': 'Media'}
)

validator_customers.expect_column_values_to_not_be_null(
    column='pais',
    meta={'dimension': 'Completitud', 'criticidad': 'Media'}
)

# Guardar suite
validator_customers.save_expectation_suite(discard_failed_expectations=False)
print('Suite de completitud creada con 4 expectativas')

### 3.2 Suite de Validez - Clientes

In [ ]:
# Crear suite de validez
suite_validez = context.add_expectation_suite(
    expectation_suite_name='customers_validez_suite'
)

validator_customers_validez = context.get_validator(
    batch_request=batch_request_customers,
    expectation_suite_name='customers_validez_suite'
)

# Expectativas de validez
validator_customers_validez.expect_column_values_to_match_regex(
    column='email',
    regex=r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$',
    mostly=0.95,
    meta={'dimension': 'Validez', 'criticidad': 'Alta'}
)

validator_customers_validez.expect_column_values_to_be_between(
    column='edad',
    min_value=18,
    max_value=100,
    meta={'dimension': 'Validez', 'criticidad': 'Alta'}
)

validator_customers_validez.expect_column_values_to_be_in_set(
    column='pais',
    value_set=['Mexico', 'Colombia', 'Argentina', 'Chile', 'Peru'],
    meta={'dimension': 'Validez', 'criticidad': 'Media'}
)

validator_customers_validez.expect_column_values_to_be_in_set(
    column='segmento',
    value_set=['Premium', 'Standard', 'Basic'],
    meta={'dimension': 'Validez', 'criticidad': 'Media'}
)

validator_customers_validez.save_expectation_suite(discard_failed_expectations=False)
print('Suite de validez creada con 4 expectativas')

### 3.3 Suite de Unicidad - Clientes

In [ ]:
# Crear suite de unicidad
suite_unicidad = context.add_expectation_suite(
    expectation_suite_name='customers_unicidad_suite'
)

validator_customers_unicidad = context.get_validator(
    batch_request=batch_request_customers,
    expectation_suite_name='customers_unicidad_suite'
)

# Expectativas de unicidad
validator_customers_unicidad.expect_column_values_to_be_unique(
    column='customer_id',
    meta={'dimension': 'Unicidad', 'criticidad': 'Alta'}
)

validator_customers_unicidad.expect_column_values_to_be_unique(
    column='email',
    mostly=0.95,
    meta={'dimension': 'Unicidad', 'criticidad': 'Alta'}
)

validator_customers_unicidad.save_expectation_suite(discard_failed_expectations=False)
print('Suite de unicidad creada con 2 expectativas')

### 3.4 Suite de Consistencia - Transacciones

In [ ]:
# Crear suite de consistencia
suite_consistencia = context.add_expectation_suite(
    expectation_suite_name='transactions_consistencia_suite'
)

validator_transactions = context.get_validator(
    batch_request=batch_request_transactions,
    expectation_suite_name='transactions_consistencia_suite'
)

# Expectativas de consistencia
validator_transactions.expect_column_values_to_be_between(
    column='cantidad',
    min_value=1,
    max_value=100,
    meta={'dimension': 'Consistencia', 'criticidad': 'Alta'}
)

validator_transactions.expect_column_values_to_be_between(
    column='precio_unitario',
    min_value=0,
    max_value=10000,
    meta={'dimension': 'Consistencia', 'criticidad': 'Alta'}
)

validator_transactions.expect_column_values_to_be_between(
    column='descuento',
    min_value=0,
    max_value=1,
    meta={'dimension': 'Consistencia', 'criticidad': 'Media'}
)

validator_transactions.expect_column_values_to_be_between(
    column='monto_total',
    min_value=0,
    max_value=100000,
    meta={'dimension': 'Consistencia', 'criticidad': 'Alta'}
)

validator_transactions.save_expectation_suite(discard_failed_expectations=False)
print('Suite de consistencia creada con 4 expectativas')

### 3.5 Suite de Puntualidad - Transacciones

In [ ]:
# Crear suite de puntualidad
suite_puntualidad = context.add_expectation_suite(
    expectation_suite_name='transactions_puntualidad_suite'
)

validator_transactions_puntualidad = context.get_validator(
    batch_request=batch_request_transactions,
    expectation_suite_name='transactions_puntualidad_suite'
)

# Expectativas de puntualidad
validator_transactions_puntualidad.expect_column_values_to_be_between(
    column='fecha_transaccion',
    min_value='2024-01-01',
    max_value=datetime.now().strftime('%Y-%m-%d'),
    meta={'dimension': 'Puntualidad', 'criticidad': 'Media'}
)

validator_transactions_puntualidad.save_expectation_suite(discard_failed_expectations=False)
print('Suite de puntualidad creada con 1 expectativa')

## Parte 4: Checkpoints Automatizados

### 4.1 Checkpoint para Clientes

In [ ]:
# Crear checkpoint para todas las suites de clientes
checkpoint_customers = context.add_checkpoint(
    name='customers_quality_checkpoint',
    validations=[
        {
            'batch_request': batch_request_customers,
            'expectation_suite_name': 'customers_completitud_suite'
        },
        {
            'batch_request': batch_request_customers,
            'expectation_suite_name': 'customers_validez_suite'
        },
        {
            'batch_request': batch_request_customers,
            'expectation_suite_name': 'customers_unicidad_suite'
        }
    ]
)

print('Checkpoint de clientes creado')

### 4.2 Checkpoint para Transacciones

In [ ]:
# Crear checkpoint para todas las suites de transacciones
checkpoint_transactions = context.add_checkpoint(
    name='transactions_quality_checkpoint',
    validations=[
        {
            'batch_request': batch_request_transactions,
            'expectation_suite_name': 'transactions_consistencia_suite'
        },
        {
            'batch_request': batch_request_transactions,
            'expectation_suite_name': 'transactions_puntualidad_suite'
        }
    ]
)

print('Checkpoint de transacciones creado')

### 4.3 Ejecutar Checkpoints

In [ ]:
# Ejecutar checkpoint de clientes
print('Ejecutando validaciones de clientes...')
result_customers = checkpoint_customers.run()

print(f'\nResultado: {"EXITOSO" if result_customers.success else "FALLIDO"}')
print(f'Total de validaciones: {len(result_customers.run_results)}')

In [ ]:
# Ejecutar checkpoint de transacciones
print('Ejecutando validaciones de transacciones...')
result_transactions = checkpoint_transactions.run()

print(f'\nResultado: {"EXITOSO" if result_transactions.success else "FALLIDO"}')
print(f'Total de validaciones: {len(result_transactions.run_results)}')

## Parte 5: Analisis de Resultados

### 5.1 Analizar Resultados por Dimension

In [ ]:
def analizar_resultados_por_dimension(checkpoint_result):
    """Analiza resultados agrupados por dimension de calidad"""
    
    dimensiones = {}
    
    for run_id, run_result in checkpoint_result.run_results.items():
        validation_result = run_result['validation_result']
        
        for result in validation_result.results:
            meta = result.expectation_config.meta or {}
            dimension = meta.get('dimension', 'Sin clasificar')
            criticidad = meta.get('criticidad', 'Media')
            
            if dimension not in dimensiones:
                dimensiones[dimension] = {
                    'total': 0,
                    'exitosas': 0,
                    'fallidas': 0,
                    'criticidad_alta': 0,
                    'criticidad_media': 0,
                    'criticidad_baja': 0
                }
            
            dimensiones[dimension]['total'] += 1
            
            if result.success:
                dimensiones[dimension]['exitosas'] += 1
            else:
                dimensiones[dimension]['fallidas'] += 1
            
            if criticidad == 'Alta':
                dimensiones[dimension]['criticidad_alta'] += 1
            elif criticidad == 'Media':
                dimensiones[dimension]['criticidad_media'] += 1
            else:
                dimensiones[dimension]['criticidad_baja'] += 1
    
    return dimensiones

# Analizar resultados de clientes
print('=== ANALISIS DE CALIDAD - CLIENTES ===')
dimensiones_customers = analizar_resultados_por_dimension(result_customers)

for dimension, stats in dimensiones_customers.items():
    tasa_exito = (stats['exitosas'] / stats['total'] * 100) if stats['total'] > 0 else 0
    print(f'\n{dimension}:')
    print(f'  Total expectativas: {stats["total"]}')
    print(f'  Exitosas: {stats["exitosas"]}')
    print(f'  Fallidas: {stats["fallidas"]}')
    print(f'  Tasa de exito: {tasa_exito:.1f}%')
    print(f'  Criticidad Alta: {stats["criticidad_alta"]}')
    print(f'  Criticidad Media: {stats["criticidad_media"]}')

In [ ]:
# Analizar resultados de transacciones
print('\n=== ANALISIS DE CALIDAD - TRANSACCIONES ===')
dimensiones_transactions = analizar_resultados_por_dimension(result_transactions)

for dimension, stats in dimensiones_transactions.items():
    tasa_exito = (stats['exitosas'] / stats['total'] * 100) if stats['total'] > 0 else 0
    print(f'\n{dimension}:')
    print(f'  Total expectativas: {stats["total"]}')
    print(f'  Exitosas: {stats["exitosas"]}')
    print(f'  Fallidas: {stats["fallidas"]}')
    print(f'  Tasa de exito: {tasa_exito:.1f}%')
    print(f'  Criticidad Alta: {stats["criticidad_alta"]}')
    print(f'  Criticidad Media: {stats["criticidad_media"]}')

### 5.2 Identificar Problemas Criticos

In [ ]:
def identificar_problemas_criticos(checkpoint_result):
    """Identifica expectativas fallidas de criticidad alta"""
    
    problemas_criticos = []
    
    for run_id, run_result in checkpoint_result.run_results.items():
        validation_result = run_result['validation_result']
        suite_name = validation_result.meta.get('expectation_suite_name', 'Desconocido')
        
        for result in validation_result.results:
            if not result.success:
                meta = result.expectation_config.meta or {}
                criticidad = meta.get('criticidad', 'Media')
                
                if criticidad == 'Alta':
                    problemas_criticos.append({
                        'suite': suite_name,
                        'expectation': result.expectation_config.expectation_type,
                        'column': result.expectation_config.kwargs.get('column', 'N/A'),
                        'dimension': meta.get('dimension', 'Sin clasificar')
                    })
    
    return problemas_criticos

# Identificar problemas criticos en clientes
print('=== PROBLEMAS CRITICOS - CLIENTES ===')
problemas_customers = identificar_problemas_criticos(result_customers)

if problemas_customers:
    for i, problema in enumerate(problemas_customers, 1):
        print(f'\n{i}. Suite: {problema["suite"]}')
        print(f'   Dimension: {problema["dimension"]}')
        print(f'   Columna: {problema["column"]}')
        print(f'   Expectativa: {problema["expectation"]}')
else:
    print('No se encontraron problemas criticos')

In [ ]:
# Identificar problemas criticos en transacciones
print('\n=== PROBLEMAS CRITICOS - TRANSACCIONES ===')
problemas_transactions = identificar_problemas_criticos(result_transactions)

if problemas_transactions:
    for i, problema in enumerate(problemas_transactions, 1):
        print(f'\n{i}. Suite: {problema["suite"]}')
        print(f'   Dimension: {problema["dimension"]}')
        print(f'   Columna: {problema["column"]}')
        print(f'   Expectativa: {problema["expectation"]}')
else:
    print('No se encontraron problemas criticos')

## Parte 6: Generacion de Reportes

### 6.1 Generar Data Docs

In [ ]:
# Generar Data Docs
context.build_data_docs()
print('Data Docs generados correctamente')
print(f'\nPuedes ver los reportes en: {gx_dir}/uncommitted/data_docs/local_site/index.html')

### 6.2 Crear Reporte Ejecutivo

In [ ]:
def generar_reporte_ejecutivo(result_customers, result_transactions):
    """Genera un reporte ejecutivo consolidado"""
    
    reporte = []
    reporte.append('=' * 80)
    reporte.append('REPORTE EJECUTIVO DE CALIDAD DE DATOS')
    reporte.append('=' * 80)
    reporte.append(f'Fecha: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    reporte.append('')
    
    # Resumen general
    total_validaciones = len(result_customers.run_results) + len(result_transactions.run_results)
    validaciones_exitosas = sum([1 for r in result_customers.run_results.values() if r['validation_result'].success])
    validaciones_exitosas += sum([1 for r in result_transactions.run_results.values() if r['validation_result'].success])
    
    reporte.append('RESUMEN GENERAL')
    reporte.append('-' * 80)
    reporte.append(f'Total de validaciones ejecutadas: {total_validaciones}')
    reporte.append(f'Validaciones exitosas: {validaciones_exitosas}')
    reporte.append(f'Validaciones fallidas: {total_validaciones - validaciones_exitosas}')
    reporte.append(f'Tasa de exito general: {(validaciones_exitosas/total_validaciones*100):.1f}%')
    reporte.append('')
    
    # Analisis por dataset
    reporte.append('ANALISIS POR DATASET')
    reporte.append('-' * 80)
    
    reporte.append('\nClientes:')
    dim_customers = analizar_resultados_por_dimension(result_customers)
    for dimension, stats in dim_customers.items():
        tasa = (stats['exitosas'] / stats['total'] * 100) if stats['total'] > 0 else 0
        reporte.append(f'  {dimension}: {stats["exitosas"]}/{stats["total"]} ({tasa:.1f}%)')
    
    reporte.append('\nTransacciones:')
    dim_transactions = analizar_resultados_por_dimension(result_transactions)
    for dimension, stats in dim_transactions.items():
        tasa = (stats['exitosas'] / stats['total'] * 100) if stats['total'] > 0 else 0
        reporte.append(f'  {dimension}: {stats["exitosas"]}/{stats["total"]} ({tasa:.1f}%)')
    
    reporte.append('')
    
    # Problemas criticos
    problemas_customers = identificar_problemas_criticos(result_customers)
    problemas_transactions = identificar_problemas_criticos(result_transactions)
    total_problemas = len(problemas_customers) + len(problemas_transactions)
    
    reporte.append('PROBLEMAS CRITICOS IDENTIFICADOS')
    reporte.append('-' * 80)
    reporte.append(f'Total de problemas criticos: {total_problemas}')
    
    if total_problemas > 0:
        reporte.append('\nAcciones recomendadas:')
        reporte.append('1. Revisar y corregir datos con criticidad alta')
        reporte.append('2. Implementar validaciones en el origen de datos')
        reporte.append('3. Establecer alertas automaticas para problemas criticos')
        reporte.append('4. Documentar y comunicar problemas al equipo responsable')
    else:
        reporte.append('\nNo se identificaron problemas criticos. Calidad de datos aceptable.')
    
    reporte.append('')
    reporte.append('=' * 80)
    
    return '\n'.join(reporte)

# Generar y mostrar reporte
reporte_ejecutivo = generar_reporte_ejecutivo(result_customers, result_transactions)
print(reporte_ejecutivo)

### 6.3 Guardar Reporte en Archivo

In [ ]:
# Guardar reporte en archivo
reporte_path = data_dir / 'reporte_calidad_datos.txt'
with open(reporte_path, 'w', encoding='utf-8') as f:
    f.write(reporte_ejecutivo)

print(f'Reporte guardado en: {reporte_path}')

## Parte 7: Automatizacion y CI/CD

### 7.1 Script de Validacion Automatizada

In [ ]:
# Crear script de validacion automatizada
script_validacion = '''#!/usr/bin/env python
"""Script de validacion automatizada para CI/CD"""

import great_expectations as gx
import sys
from pathlib import Path

def main():
    # Inicializar contexto
    gx_dir = Path('../gx_proyecto_integrador')
    context = gx.get_context(mode='file', project_root_dir=str(gx_dir))
    
    # Ejecutar checkpoints
    print('Ejecutando validaciones de calidad...')
    
    checkpoint_customers = context.get_checkpoint('customers_quality_checkpoint')
    result_customers = checkpoint_customers.run()
    
    checkpoint_transactions = context.get_checkpoint('transactions_quality_checkpoint')
    result_transactions = checkpoint_transactions.run()
    
    # Verificar resultados
    all_success = result_customers.success and result_transactions.success
    
    if all_success:
        print('VALIDACION EXITOSA: Todos los checkpoints pasaron')
        sys.exit(0)
    else:
        print('VALIDACION FALLIDA: Algunos checkpoints no pasaron')
        sys.exit(1)

if __name__ == '__main__':
    main()
'''

# Guardar script
script_path = data_dir / 'validar_calidad.py'
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(script_validacion)

print(f'Script de validacion creado en: {script_path}')

### 7.2 Configuracion de GitHub Actions

In [ ]:
# Crear configuracion de GitHub Actions
github_actions_config = '''name: Data Quality Validation

on:
  push:
    branches: [ main, develop ]
  pull_request:
    branches: [ main ]
  schedule:
    - cron: '0 8 * * *'  # Ejecutar diariamente a las 8 AM

jobs:
  validate-data-quality:
    runs-on: ubuntu-latest
    
    steps:
    - uses: actions/checkout@v3
    
    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.10'
    
    - name: Install dependencies
      run: |
        pip install great-expectations pandas numpy
    
    - name: Run data quality validations
      run: |
        python data/proyecto_integrador/validar_calidad.py
    
    - name: Upload Data Docs
      if: always()
      uses: actions/upload-artifact@v3
      with:
        name: data-docs
        path: gx_proyecto_integrador/uncommitted/data_docs/
'''

# Guardar configuracion
github_path = data_dir / 'github_actions_workflow.yml'
with open(github_path, 'w', encoding='utf-8') as f:
    f.write(github_actions_config)

print(f'Configuracion de GitHub Actions creada en: {github_path}')

## Parte 8: Sistema de Alertas

### 8.1 Funcion de Alertas por Email (Simulacion)

In [ ]:
def enviar_alerta_email(problemas_criticos, destinatarios):
    """Simula el envio de alertas por email"""
    
    if not problemas_criticos:
        print('No hay problemas criticos. No se enviaran alertas.')
        return
    
    print('\n--- SIMULACION DE ALERTA POR EMAIL ---')
    print(f'Para: {destinatarios}')
    print('Asunto: ALERTA - Problemas Criticos de Calidad de Datos Detectados')
    print('\nCuerpo del mensaje:')
    print('\nSe han detectado problemas criticos en la validacion de calidad de datos:')
    print(f'\nTotal de problemas criticos: {len(problemas_criticos)}')
    print('\nDetalle de problemas:')
    
    for i, problema in enumerate(problemas_criticos, 1):
        print(f'\n{i}. {problema["dimension"]} - {problema["column"]}')
        print(f'   Suite: {problema["suite"]}')
        print(f'   Expectativa: {problema["expectation"]}')
    
    print('\nPor favor, revise los Data Docs para mas detalles.')
    print('\n--- FIN DE SIMULACION ---')

# Simular envio de alertas
todos_problemas = problemas_customers + problemas_transactions
enviar_alerta_email(
    problemas_criticos=todos_problemas,
    destinatarios='equipo-datos@empresa.com'
)

### 8.2 Integracion con Slack (Simulacion)

In [ ]:
def enviar_alerta_slack(checkpoint_result, canal='#data-quality'):
    """Simula el envio de alertas a Slack"""
    
    total_validaciones = len(checkpoint_result.run_results)
    validaciones_exitosas = sum([1 for r in checkpoint_result.run_results.values() 
                                  if r['validation_result'].success])
    tasa_exito = (validaciones_exitosas / total_validaciones * 100) if total_validaciones > 0 else 0
    
    print('\n--- SIMULACION DE MENSAJE SLACK ---')
    print(f'Canal: {canal}')
    print('\nMensaje:')
    
    if checkpoint_result.success:
        print('Validacion de Calidad de Datos - EXITOSA')
    else:
        print('Validacion de Calidad de Datos - FALLIDA')
    
    print(f'\nResultados:')
    print(f'- Total validaciones: {total_validaciones}')
    print(f'- Exitosas: {validaciones_exitosas}')
    print(f'- Fallidas: {total_validaciones - validaciones_exitosas}')
    print(f'- Tasa de exito: {tasa_exito:.1f}%')
    print('\n--- FIN DE SIMULACION ---')

# Simular alertas de Slack
enviar_alerta_slack(result_customers, canal='#data-quality-customers')
enviar_alerta_slack(result_transactions, canal='#data-quality-transactions')

## Parte 9: Metricas y KPIs de Calidad

### 9.1 Calcular KPIs de Calidad

In [ ]:
def calcular_kpis_calidad(result_customers, result_transactions):
    """Calcula KPIs clave de calidad de datos"""
    
    kpis = {}
    
    # KPI 1: Tasa de exito general
    total_validaciones = len(result_customers.run_results) + len(result_transactions.run_results)
    validaciones_exitosas = sum([1 for r in result_customers.run_results.values() 
                                  if r['validation_result'].success])
    validaciones_exitosas += sum([1 for r in result_transactions.run_results.values() 
                                   if r['validation_result'].success])
    
    kpis['tasa_exito_general'] = (validaciones_exitosas / total_validaciones * 100) if total_validaciones > 0 else 0
    
    # KPI 2: Problemas criticos por dataset
    kpis['problemas_criticos_customers'] = len(identificar_problemas_criticos(result_customers))
    kpis['problemas_criticos_transactions'] = len(identificar_problemas_criticos(result_transactions))
    kpis['problemas_criticos_total'] = kpis['problemas_criticos_customers'] + kpis['problemas_criticos_transactions']
    
    # KPI 3: Cobertura de dimensiones
    dimensiones_customers = set(analizar_resultados_por_dimension(result_customers).keys())
    dimensiones_transactions = set(analizar_resultados_por_dimension(result_transactions).keys())
    dimensiones_cubiertas = dimensiones_customers.union(dimensiones_transactions)
    
    kpis['dimensiones_cubiertas'] = len(dimensiones_cubiertas)
    kpis['cobertura_dimensiones'] = (len(dimensiones_cubiertas) / 5 * 100)  # 5 dimensiones totales
    
    # KPI 4: Tasa de exito por dimension
    kpis['tasas_por_dimension'] = {}
    
    for dimension in dimensiones_cubiertas:
        total = 0
        exitosas = 0
        
        dim_customers = analizar_resultados_por_dimension(result_customers)
        if dimension in dim_customers:
            total += dim_customers[dimension]['total']
            exitosas += dim_customers[dimension]['exitosas']
        
        dim_transactions = analizar_resultados_por_dimension(result_transactions)
        if dimension in dim_transactions:
            total += dim_transactions[dimension]['total']
            exitosas += dim_transactions[dimension]['exitosas']
        
        kpis['tasas_por_dimension'][dimension] = (exitosas / total * 100) if total > 0 else 0
    
    return kpis

# Calcular KPIs
kpis = calcular_kpis_calidad(result_customers, result_transactions)

print('=== KPIs DE CALIDAD DE DATOS ===')
print(f'\nTasa de Exito General: {kpis["tasa_exito_general"]:.1f}%')
print(f'Problemas Criticos Totales: {kpis["problemas_criticos_total"]}')
print(f'  - Clientes: {kpis["problemas_criticos_customers"]}')
print(f'  - Transacciones: {kpis["problemas_criticos_transactions"]}')
print(f'\nCobertura de Dimensiones: {kpis["cobertura_dimensiones"]:.0f}% ({kpis["dimensiones_cubiertas"]}/5)')
print('\nTasa de Exito por Dimension:')
for dimension, tasa in kpis['tasas_por_dimension'].items():
    print(f'  {dimension}: {tasa:.1f}%')

### 9.2 Visualizacion de KPIs

In [ ]:
import matplotlib.pyplot as plt

# Crear visualizaciones de KPIs
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Dashboard de Calidad de Datos', fontsize=16, fontweight='bold')

# Grafico 1: Tasa de exito general
ax1 = axes[0, 0]
tasa_exito = kpis['tasa_exito_general']
tasa_fallo = 100 - tasa_exito
ax1.pie([tasa_exito, tasa_fallo], labels=['Exitosas', 'Fallidas'], 
        autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
ax1.set_title('Tasa de Exito General')

# Grafico 2: Problemas criticos por dataset
ax2 = axes[0, 1]
datasets = ['Clientes', 'Transacciones']
problemas = [kpis['problemas_criticos_customers'], kpis['problemas_criticos_transactions']]
ax2.bar(datasets, problemas, color=['#3498db', '#9b59b6'])
ax2.set_title('Problemas Criticos por Dataset')
ax2.set_ylabel('Numero de Problemas')
ax2.grid(axis='y', alpha=0.3)

# Grafico 3: Tasa de exito por dimension
ax3 = axes[1, 0]
dimensiones = list(kpis['tasas_por_dimension'].keys())
tasas = list(kpis['tasas_por_dimension'].values())
colors_dim = ['#e74c3c' if t < 80 else '#f39c12' if t < 95 else '#2ecc71' for t in tasas]
ax3.barh(dimensiones, tasas, color=colors_dim)
ax3.set_title('Tasa de Exito por Dimension')
ax3.set_xlabel('Tasa de Exito (%)')
ax3.set_xlim(0, 100)
ax3.grid(axis='x', alpha=0.3)

# Grafico 4: Cobertura de dimensiones
ax4 = axes[1, 1]
dimensiones_totales = ['Completitud', 'Validez', 'Unicidad', 'Consistencia', 'Puntualidad']
cobertura = [1 if d in dimensiones else 0 for d in dimensiones_totales]
colors_cob = ['#2ecc71' if c == 1 else '#ecf0f1' for c in cobertura]
ax4.bar(dimensiones_totales, cobertura, color=colors_cob)
ax4.set_title('Cobertura de Dimensiones')
ax4.set_ylabel('Implementada (1) / No Implementada (0)')
ax4.set_ylim(0, 1.2)
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(data_dir / 'dashboard_calidad.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Dashboard guardado en: {data_dir / "dashboard_calidad.png"}')

## Parte 10: Conclusiones y Siguientes Pasos

### 10.1 Resumen del Proyecto

In [ ]:
print('=' * 80)
print('RESUMEN DEL PROYECTO INTEGRADOR')
print('=' * 80)
print('\nLo que has logrado en este proyecto:')
print('\n1. Configuracion de un sistema completo de calidad de datos')
print('   - File Context persistente')
print('   - Multiples data sources (CSV, Parquet)')
print('\n2. Implementacion de las 5 dimensiones de calidad')
print('   - Completitud')
print('   - Validez')
print('   - Unicidad')
print('   - Consistencia')
print('   - Puntualidad')
print('\n3. Automatizacion con Checkpoints')
print('   - Checkpoints para clientes')
print('   - Checkpoints para transacciones')
print('\n4. Reportes y visualizaciones')
print('   - Data Docs')
print('   - Reporte ejecutivo')
print('   - Dashboard de KPIs')
print('\n5. Integracion CI/CD')
print('   - Script de validacion automatizada')
print('   - Configuracion de GitHub Actions')
print('\n6. Sistema de alertas')
print('   - Alertas por email')
print('   - Integracion con Slack')
print('\n' + '=' * 80)

### 10.2 Siguientes Pasos Recomendados

In [ ]:
print('SIGUIENTES PASOS PARA LLEVAR ESTE PROYECTO A PRODUCCION:')
print('\n1. Conectar a fuentes de datos reales')
print('   - Configurar conexiones a bases de datos de produccion')
print('   - Implementar autenticacion y seguridad')
print('   - Configurar data sources para S3, Azure Blob, etc.')
print('\n2. Expandir expectativas')
print('   - Agregar mas reglas de negocio especificas')
print('   - Implementar expectativas personalizadas')
print('   - Incluir validaciones de integridad referencial')
print('\n3. Implementar alertas reales')
print('   - Configurar SMTP para emails')
print('   - Integrar con Slack usando webhooks')
print('   - Configurar PagerDuty para incidentes criticos')
print('\n4. Optimizar rendimiento')
print('   - Implementar validaciones incrementales')
print('   - Usar sampling para datasets grandes')
print('   - Paralelizar ejecucion de checkpoints')
print('\n5. Establecer gobernanza')
print('   - Definir SLAs de calidad de datos')
print('   - Crear proceso de revision de expectativas')
print('   - Documentar responsabilidades del equipo')
print('\n6. Monitoreo continuo')
print('   - Configurar dashboards en tiempo real')
print('   - Implementar tracking de tendencias')
print('   - Establecer metricas de mejora continua')

### 10.3 Recursos Adicionales

In [ ]:
print('RECURSOS PARA CONTINUAR APRENDIENDO:')
print('\n1. Documentacion Oficial')
print('   - https://docs.greatexpectations.io/')
print('   - https://greatexpectations.io/blog/')
print('\n2. Comunidad')
print('   - Slack de Great Expectations')
print('   - GitHub Discussions')
print('   - Stack Overflow')
print('\n3. Ejemplos y Casos de Uso')
print('   - Great Expectations Gallery')
print('   - GitHub Examples Repository')
print('   - Case Studies en el blog oficial')
print('\n4. Integraciones')
print('   - Airflow + Great Expectations')
print('   - dbt + Great Expectations')
print('   - Databricks + Great Expectations')
print('   - Snowflake + Great Expectations')

## Ejercicios Adicionales (Opcional)

### Ejercicio 1: Agregar Validacion de Base de Datos

Conecta a una base de datos PostgreSQL y crea expectativas para validar:
- Integridad referencial entre tablas
- Constraints de base de datos
- Queries SQL personalizadas

### Ejercicio 2: Implementar Expectativas Personalizadas

Crea una expectativa personalizada que valide:
- Formato de numeros de telefono mexicanos
- Codigos postales validos
- RFC validos

### Ejercicio 3: Dashboard en Tiempo Real

Implementa un dashboard usando Streamlit o Dash que muestre:
- Metricas de calidad en tiempo real
- Historico de validaciones
- Alertas activas

### Ejercicio 4: Integracion con Airflow

Crea un DAG de Airflow que:
- Ejecute validaciones despues de cada ETL
- Envie alertas en caso de fallo
- Genere reportes diarios

### Ejercicio 5: Validacion de Data Drift

Implementa validaciones que detecten:
- Cambios en distribuciones estadisticas
- Nuevos valores en columnas categoricas
- Cambios en volumenes de datos

## Felicitaciones

Has completado el proyecto integrador del taller de Great Expectations.

Ahora tienes las herramientas y conocimientos para implementar un sistema robusto de calidad de datos en tu organizacion.

Recuerda: La calidad de datos es un proceso continuo, no un proyecto unico. Sigue iterando, mejorando y adaptando tus validaciones a medida que evolucionan tus datos y necesidades de negocio.